# 3. Predict - simulated CRM data (`data/crm_simulated/`)

Same step like `python -m src.predict --output reports/crm/simulated/final`, using function in `src/predict.py`.
It use the frozen calibrated model which `1_train.ipynb` was saved. Open deals is never get labeled as Lost.

Score on simulated deals is only for showing the workflow.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").is_file())
sys.path.insert(0, str(ROOT))

import tempfile
import pandas as pd
from src import predict, train
from src.outputs import Outputs

## Settings

Need to be same setting as in `1_train.ipynb`.

In [ ]:
DATA_DIR = ROOT / "data/crm_simulated"
QUICK = False
OUTPUT = train.default_output(DATA_DIR, QUICK)
print("Data:  ", DATA_DIR)
print("Output:", OUTPUT)

## 1. Load the scoring bundle

In [ ]:
out = Outputs(OUTPUT)
scoring = predict.load(out)
print("Model:", scoring["model"].name)
print("Decision threshold (calibrated):", round(scoring["decision_threshold"], 4))
print("Features:", len(scoring["features"]), "| data:", out.source_data())

## 2. Score the open Engaging deals

This write `predictions/open_deal_predictions.csv` with P(Won), P(Lost), the predicted class, a priority band (heuristic),
the input that push each score up or down, and a warning if the account is missing.

In [ ]:
scores = predict.run(OUTPUT)
pd.Series({
    "scored deals": len(scores),
    "predicted Won": int(scores.predicted_outcome.eq("Won").sum()),
    "High / Medium / Low": "/".join(str(int(scores.priority.eq(p).sum())) for p in ["High", "Medium", "Low"]),
    "missing account (warning)": int(scores.account_missing.sum()),
    "mean P(Won)": round(scores.win_probability.mean(), 4),
})

In [ ]:
scores[["opportunity_id", "product", "sales_agent", "win_probability", "priority",
        "positive_factors", "negative_factors", "input_warning"]].head(10)

## 3. Score new deal from CSV

New deals need same format like `sales_pipeline.csv`: `opportunity_id, sales_agent, product, account, engage_date`.
Here we just take five Engaging deal from the data for example, you can replacing `new_deals` with you own file.

In [ ]:
raw = pd.read_csv(out.source_data() / "sales_pipeline.csv")
new_deals = raw.loc[raw.deal_stage.eq("Engaging") & raw.account.notna(),
                    ["opportunity_id", "sales_agent", "product", "account", "engage_date"]].head(5)
with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "new_deals.csv"
    new_deals.to_csv(path, index=False)
    new_scores = predict.score_file(path, OUTPUT, Path(folder) / "new_deal_scores.csv")
new_scores[["opportunity_id", "win_probability", "predicted_outcome", "priority", "positive_factors"]]